# NumPy and Fast pandas

Notebook for **Chapter 6** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch06.html

Tip: change a number, predict the output, then run the cell again.

## 6.1 NumPy Arrays: the Engine Under pandas

In [ ]:
import numpy as np
a = np.array([[1, 2, 3], [4, 5, 6]])
print(a.shape, a.dtype, a.ndim, a.size)
print(a.sum(), a.sum(axis=0), a.sum(axis=1), a.mean(axis=0))

In [ ]:
print(a[:, 1], a[1, :2], a[a > 2])

[2 5] [4 5] [3 4 5 6]


## 6.2 Broadcasting, dtypes and Random Numbers

In [ ]:
print(a + np.array([10, 20, 30]))
print(a + np.array([[100], [200]]))

[[11 22 33]
 [14 25 36]]
[[101 102 103]
 [204 205 206]]


In [ ]:
print(np.array([1, 2, 3], dtype=np.int8) * 100)
print(0.1 + 0.2 == 0.3, np.isclose(0.1 + 0.2, 0.3))

[100 -56  44]
False True


In [ ]:
rng = np.random.default_rng(42)
print(rng.integers(1, 7, size=5), rng.normal(100, 15, size=3).round(2))

[1 5 4 3 3] [114.11  70.73  80.47]


In [ ]:
A = np.array([[2., 1.], [1., 3.]]); b = np.array([5., 10.])
print(np.linalg.solve(A, b), np.linalg.inv(A).round(3), np.linalg.det(A), A @ A)

[1. 3.] [[ 0.6 -0.2]
 [-0.2  0.4]] 5.000000000000001 [[ 5.  5.]
 [ 5. 10.]]


## 6.3 Vectorisation: Stop Writing Loops

In [ ]:
import time
v = rng.normal(size=1_000_000)
t = time.perf_counter(); s = 0.0
for e in v: s += e * e
t1 = time.perf_counter() - t
t = time.perf_counter(); s2 = (v * v).sum(); t2 = time.perf_counter() - t
print(f"loop {t1:.3f}s  vector {t2:.5f}s  speedup {t1/t2:.0f}x", np.isclose(s, s2))

In [ ]:
n = 300_000
df = pd.DataFrame({"a": rng.integers(0, 100, n), "b": rng.random(n)})
r1 = df.apply(lambda r: r.a * r.b, axis=1)    # row by row
r2 = df.a * df.b                              # whole column

apply 1.60s vectorized 0.0008s True


## 6.4 Memory: dtypes, category, chunking

In [ ]:
cities = rng.choice(["Pune", "Delhi", "Mumbai", "Chennai"], size=n)
d2 = pd.DataFrame({"city": cities})
m1 = d2.memory_usage(deep=True).sum()
d2["city"] = d2.city.astype("category")
m2 = d2.memory_usage(deep=True).sum()
print(m1, m2, round(m1 / m2, 1))

16349930 300350 54.4


## 6.5 Beyond pandas: Polars and DuckDB (awareness)

In [ ]:
import polars as pl
lf = (pl.scan_csv("orders.csv")                    # lazy: nothing is read yet
        .filter(pl.col("status") == "delivered")
        .group_by("city")
        .agg(pl.col("amount").sum().alias("revenue"))
        .sort("revenue", descending=True))
print(lf.collect())                                # runs the whole plan now

In [ ]:
import duckdb
duckdb.sql("""
    SELECT city, SUM(amount) AS revenue
    FROM 'orders.csv'
    WHERE status = 'delivered'
    GROUP BY city
    ORDER BY revenue DESC
""").df()                                          # returns a pandas DataFrame

## 6.6 SQL for Analysts: Core Queries

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers(cust_id INT, name TEXT, city TEXT, signup_date TEXT);
INSERT INTO customers VALUES
 (1,'Asha','Pune','2025-01-05'),(2,'Ravi','Delhi','2025-01-20'),
 (3,'Meera','Pune','2025-02-03'),(4,'Imran','Mumbai','2025-02-14'),
 (5,'Kavya','Delhi','2025-03-01'),(6,'Dev','Mumbai','2025-03-10');
CREATE TABLE orders(order_id INT, cust_id INT, order_date TEXT, amount REAL, status TEXT);
INSERT INTO orders VALUES
 (1,1,'2025-01-06',1200,'delivered'),(2,1,'2025-01-25',800,'delivered'),
 (3,2,'2025-01-22',1500,'delivered'),(4,3,'2025-02-04',500,'returned'),
 (5,3,'2025-02-20',900,'delivered'),(6,1,'2025-02-11',2000,'delivered'),
 (7,4,'2025-02-15',700,'delivered'),(8,2,'2025-03-02',1100,'delivered'),
 (9,5,'2025-03-03',NULL,'pending'),(10,4,'2025-03-12',1300,'delivered'),
 (11,1,'2025-03-15',600,'delivered'),(12,5,'2025-03-20',2500,'delivered'),
 (13,3,'2025-03-22',400,'cancelled'),(14,2,'2025-04-02',1700,'delivered');
""")
def q(sql):
    print(pd.read_sql(sql, con).to_string(index=False)); print()

# 1. one row per city
q("SELECT city, COUNT(*) AS customers FROM customers GROUP BY city ORDER BY customers DESC, city")
# 2. COUNT(*) counts rows, COUNT(amount) skips NULLs, AVG skips NULLs too
q("""SELECT status, COUNT(*) AS n, SUM(amount) AS total, COUNT(amount) AS n_amount, AVG(amount) AS avg_amt
     FROM orders GROUP BY status""")
# 3. WHERE filters rows first, HAVING filters buckets after SUM
q("""SELECT cust_id, SUM(amount) AS spend FROM orders
     WHERE status = 'delivered' GROUP BY cust_id
     HAVING SUM(amount) > 2000 ORDER BY spend DESC""")

  city  customers
 Delhi          2
Mumbai          2
  Pune          2

   status  n   total  n_amount  avg_amt
cancelled  1   400.0         1    400.0
delivered 11 14300.0        11   1300.0
  pending  1     NaN         0      NaN
 returned  1   500.0         1    500.0

 cust_id  spend
       1 4600.0
       2 4300.0
       5 2500.0


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers(cust_id INT, name TEXT, city TEXT, signup_date TEXT);
INSERT INTO customers VALUES
 (1,'Asha','Pune','2025-01-05'),(2,'Ravi','Delhi','2025-01-20'),
 (3,'Meera','Pune','2025-02-03'),(4,'Imran','Mumbai','2025-02-14'),
 (5,'Kavya','Delhi','2025-03-01'),(6,'Dev','Mumbai','2025-03-10');
CREATE TABLE orders(order_id INT, cust_id INT, order_date TEXT, amount REAL, status TEXT);
INSERT INTO orders VALUES
 (1,1,'2025-01-06',1200,'delivered'),(2,1,'2025-01-25',800,'delivered'),
 (3,2,'2025-01-22',1500,'delivered'),(4,3,'2025-02-04',500,'returned'),
 (5,3,'2025-02-20',900,'delivered'),(6,1,'2025-02-11',2000,'delivered'),
 (7,4,'2025-02-15',700,'delivered'),(8,2,'2025-03-02',1100,'delivered'),
 (9,5,'2025-03-03',NULL,'pending'),(10,4,'2025-03-12',1300,'delivered'),
 (11,1,'2025-03-15',600,'delivered'),(12,5,'2025-03-20',2500,'delivered'),
 (13,3,'2025-03-22',400,'cancelled'),(14,2,'2025-04-02',1700,'delivered');
""")
def q(sql):
    print(pd.read_sql(sql, con).to_string(index=False)); print()

# 1. inner join: Mumbai customers who ordered
q("""SELECT c.name, o.order_id, o.amount FROM customers c
     INNER JOIN orders o ON o.cust_id = c.cust_id WHERE c.city = 'Mumbai'""")
# 2. left join: every customer, with 0 for no orders
q("""SELECT c.name, COUNT(o.order_id) AS n_orders, COALESCE(SUM(o.amount), 0) AS spend
     FROM customers c LEFT JOIN orders o ON o.cust_id = c.cust_id
     GROUP BY c.name ORDER BY spend DESC, c.name""")
# 3. anti-join: customers who never ordered
q("""SELECT c.name FROM customers c
     LEFT JOIN orders o ON o.cust_id = c.cust_id WHERE o.order_id IS NULL""")
# 4. where the filter lives changes the answer
q("""SELECT COUNT(*) AS filter_in_where FROM customers c
     LEFT JOIN orders o ON o.cust_id = c.cust_id WHERE o.status = 'delivered'""")
q("""SELECT COUNT(*) AS filter_in_on FROM customers c
     LEFT JOIN orders o ON o.cust_id = c.cust_id AND o.status = 'delivered'""")

 name  order_id  amount
Imran         7   700.0
Imran        10  1300.0

 name  n_orders  spend
 Asha         4 4600.0
 Ravi         3 4300.0
Kavya         2 2500.0
Imran         2 2000.0
Meera         3 1800.0
  Dev         0    0.0

name
 Dev

 filter_in_where
              11

 filter_in_on
           12


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers(cust_id INT, name TEXT, city TEXT, signup_date TEXT);
INSERT INTO customers VALUES
 (1,'Asha','Pune','2025-01-05'),(2,'Ravi','Delhi','2025-01-20'),
 (3,'Meera','Pune','2025-02-03'),(4,'Imran','Mumbai','2025-02-14'),
 (5,'Kavya','Delhi','2025-03-01'),(6,'Dev','Mumbai','2025-03-10');
CREATE TABLE orders(order_id INT, cust_id INT, order_date TEXT, amount REAL, status TEXT);
INSERT INTO orders VALUES
 (1,1,'2025-01-06',1200,'delivered'),(2,1,'2025-01-25',800,'delivered'),
 (3,2,'2025-01-22',1500,'delivered'),(4,3,'2025-02-04',500,'returned'),
 (5,3,'2025-02-20',900,'delivered'),(6,1,'2025-02-11',2000,'delivered'),
 (7,4,'2025-02-15',700,'delivered'),(8,2,'2025-03-02',1100,'delivered'),
 (9,5,'2025-03-03',NULL,'pending'),(10,4,'2025-03-12',1300,'delivered'),
 (11,1,'2025-03-15',600,'delivered'),(12,5,'2025-03-20',2500,'delivered'),
 (13,3,'2025-03-22',400,'cancelled'),(14,2,'2025-04-02',1700,'delivered');
""")
def q(sql):
    print(pd.read_sql(sql, con).to_string(index=False)); print()

# 1. CTEs: share of delivered revenue per customer
q("""WITH delivered AS (SELECT * FROM orders WHERE status = 'delivered'),
          spend AS (SELECT cust_id, SUM(amount) AS spend FROM delivered GROUP BY cust_id)
     SELECT c.name, s.spend, ROUND(s.spend * 100.0 / (SELECT SUM(spend) FROM spend), 1) AS pct
     FROM spend s JOIN customers c USING (cust_id) ORDER BY s.spend DESC""")
# 2. CASE: bands (the order of WHEN lines matters, first match wins)
q("""SELECT band, COUNT(*) AS n FROM (
       SELECT CASE WHEN amount IS NULL THEN 'unknown' WHEN amount >= 1500 THEN 'large'
                   WHEN amount >= 800 THEN 'medium' ELSE 'small' END AS band
       FROM orders) GROUP BY band ORDER BY n DESC, band""")
# 3. scalar subquery: orders above the overall average
q("SELECT order_id, amount FROM orders WHERE amount > (SELECT AVG(amount) FROM orders) ORDER BY order_id")

 name  spend  pct
 Asha 4600.0 32.2
 Ravi 4300.0 30.1
Kavya 2500.0 17.5
Imran 2000.0 14.0
Meera  900.0  6.3

   band  n
 medium  5
  large  4
  small  4
unknown  1

 order_id  amount
        1  1200.0
        3  1500.0
        6  2000.0
       10  1300.0
       12  2500.0
       14  1700.0


In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers(cust_id INT, name TEXT);
INSERT INTO customers VALUES (1,'Asha'),(2,'Ravi'),(6,'Dev');
CREATE TABLE orders(order_id INT, cust_id INT, amount REAL);
INSERT INTO orders VALUES (1,1,1200),(2,2,NULL),(3,NULL,400);
""")
def one(sql):
    print(sql.split("--")[0].strip(), "->", con.execute(sql).fetchall())

one("SELECT NULL = NULL, NULL IS NULL, 1 + NULL, COALESCE(NULL, 0), NULLIF(5, 5)")
one("SELECT COUNT(*) FROM orders WHERE amount < 1000 OR amount >= 1000")      # misses the NULL row
one("SELECT COUNT(*) FROM orders WHERE amount < 1000 OR amount >= 1000 OR amount IS NULL")
# the NOT IN trap: one NULL in the list makes every answer unknown
one("SELECT name FROM customers WHERE cust_id NOT IN (SELECT cust_id FROM orders)")
one("""SELECT name FROM customers c WHERE NOT EXISTS
       (SELECT 1 FROM orders o WHERE o.cust_id = c.cust_id)""")

SELECT NULL = NULL, NULL IS NULL, 1 + NULL, COALESCE(NULL, 0), NULLIF(5, 5) -> [(None, 1, None, 0, None)]
SELECT COUNT(*) FROM orders WHERE amount < 1000 OR amount >= 1000 -> [(2,)]
SELECT COUNT(*) FROM orders WHERE amount < 1000 OR amount >= 1000 OR amount IS NULL -> [(3,)]
SELECT name FROM customers WHERE cust_id NOT IN (SELECT cust_id FROM orders) -> []
SELECT name FROM customers c WHERE NOT EXISTS
       (SELECT 1 FROM orders o WHERE o.cust_id = c.cust_id) -> [('Dev',)]


## 6.7 SQL Window Functions

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers(cust_id INT, name TEXT, city TEXT, signup_date TEXT);
INSERT INTO customers VALUES
 (1,'Asha','Pune','2025-01-05'),(2,'Ravi','Delhi','2025-01-20'),
 (3,'Meera','Pune','2025-02-03'),(4,'Imran','Mumbai','2025-02-14'),
 (5,'Kavya','Delhi','2025-03-01'),(6,'Dev','Mumbai','2025-03-10');
CREATE TABLE orders(order_id INT, cust_id INT, order_date TEXT, amount REAL, status TEXT);
INSERT INTO orders VALUES
 (1,1,'2025-01-06',1200,'delivered'),(2,1,'2025-01-25',800,'delivered'),
 (3,2,'2025-01-22',1500,'delivered'),(4,3,'2025-02-04',500,'returned'),
 (5,3,'2025-02-20',900,'delivered'),(6,1,'2025-02-11',2000,'delivered'),
 (7,4,'2025-02-15',700,'delivered'),(8,2,'2025-03-02',1100,'delivered'),
 (9,5,'2025-03-03',NULL,'pending'),(10,4,'2025-03-12',1300,'delivered'),
 (11,1,'2025-03-15',600,'delivered'),(12,5,'2025-03-20',2500,'delivered'),
 (13,3,'2025-03-22',400,'cancelled'),(14,2,'2025-04-02',1700,'delivered');
""")
def q(sql):
    print(pd.read_sql(sql, con).to_string(index=False, float_format=lambda v: f"{v:.1f}")); print()

# 1. order number per customer, and overall rank by amount
q("""SELECT order_id, cust_id, amount,
            ROW_NUMBER() OVER (PARTITION BY cust_id ORDER BY order_date) AS nth_order,
            RANK() OVER (ORDER BY amount DESC) AS amount_rank
     FROM orders WHERE status = 'delivered' ORDER BY cust_id, order_date""")
# 2. days since the customer's previous order
q("""SELECT cust_id, order_date, amount,
            LAG(order_date) OVER w AS prev_date,
            CAST(julianday(order_date) - julianday(LAG(order_date) OVER w) AS INT) AS gap_days
     FROM orders WHERE status = 'delivered'
     WINDOW w AS (PARTITION BY cust_id ORDER BY order_date)
     ORDER BY cust_id, order_date""")
# 3. running total and 3-order moving average
q("""SELECT order_date, amount,
            SUM(amount) OVER (ORDER BY order_date, order_id) AS running_total,
            AVG(amount) OVER (ORDER BY order_date, order_id
                              ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS ma3
     FROM orders WHERE status = 'delivered' ORDER BY order_date, order_id""")

 order_id  cust_id  amount  nth_order  amount_rank
        1        1  1200.0          1            6
        2        1   800.0          2            9
        6        1  2000.0          3            2
       11        1   600.0          4           11
        3        2  1500.0          1            4
        8        2  1100.0          2            7
       14        2  1700.0          3            3
        5        3   900.0          1            8
        7        4   700.0          1           10
       10        4  1300.0          2            5
       12        5  2500.0          1            1

 cust_id order_date  amount  prev_date  gap_days
       1 2025-01-06  1200.0        NaN       NaN
       1 2025-01-25   800.0 2025-01-06      19.0
       1 2025-02-11  2000.0 2025-01-25      17.0
       1 2025-03-15   600.0 2025-02-11      32.0
       2 2025-01-22  1500.0        NaN       NaN
       2 2025-03-02  1100.0 2025-01-22      39.0
       2 2025-04-02  1700.0 2025-03-02      

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE m(year INT, mon INT, revenue REAL);
INSERT INTO m VALUES (2024,1,3500),(2024,2,4000),(2024,3,4400),
                     (2025,1,4800),(2025,2,5200),(2025,3,6000);
""")
sql = """
SELECT a.year, a.mon, a.revenue,
       ROUND((a.revenue / LAG(a.revenue) OVER (ORDER BY a.year, a.mon) - 1) * 100, 1) AS mom_pct,
       ROUND((a.revenue / b.revenue - 1) * 100, 1) AS yoy_pct
FROM m a LEFT JOIN m b ON b.year = a.year - 1 AND b.mon = a.mon
ORDER BY a.year, a.mon"""
print(pd.read_sql(sql, con).to_string(index=False))

 year  mon  revenue  mom_pct  yoy_pct
 2024    1   3500.0      NaN      NaN
 2024    2   4000.0     14.3      NaN
 2024    3   4400.0     10.0      NaN
 2025    1   4800.0      9.1     37.1
 2025    2   5200.0      8.3     30.0
 2025    3   6000.0     15.4     36.4


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers(cust_id INT, name TEXT, city TEXT);
INSERT INTO customers VALUES (1,'Asha','Pune'),(2,'Ravi','Delhi'),(3,'Meera','Pune'),
                             (4,'Imran','Mumbai'),(5,'Kavya','Delhi'),(6,'Dev','Mumbai');
CREATE TABLE orders(order_id INT, cust_id INT, amount REAL, status TEXT);
INSERT INTO orders VALUES
 (1,1,1200,'delivered'),(2,1,800,'delivered'),(3,2,1500,'delivered'),(5,3,900,'delivered'),
 (6,1,2000,'delivered'),(7,4,700,'delivered'),(8,2,1100,'delivered'),(10,4,1300,'delivered'),
 (11,1,600,'delivered'),(12,5,2500,'delivered'),(14,2,1700,'delivered');
""")
sql = """
WITH ranked AS (
  SELECT c.city, o.order_id, o.amount,
         ROW_NUMBER() OVER (PARTITION BY c.city ORDER BY o.amount DESC) AS rn
  FROM orders o JOIN customers c USING (cust_id) WHERE o.status = 'delivered')
SELECT city, order_id, amount, rn FROM ranked WHERE rn <= 2 ORDER BY city, rn"""
print(pd.read_sql(sql, con).to_string(index=False))

# ties: which ranking function do you want?
sql2 = """SELECT x, ROW_NUMBER() OVER (ORDER BY x DESC) AS row_number,
                 RANK() OVER (ORDER BY x DESC) AS rank,
                 DENSE_RANK() OVER (ORDER BY x DESC) AS dense_rank
          FROM (SELECT 90 AS x UNION ALL SELECT 80 UNION ALL SELECT 80 UNION ALL SELECT 70)"""
print(pd.read_sql(sql2, con).to_string(index=False))

  city  order_id  amount  rn
 Delhi        12  2500.0   1
 Delhi        14  1700.0   2
Mumbai        10  1300.0   1
Mumbai         7   700.0   2
  Pune         6  2000.0   1
  Pune         1  1200.0   2
 x  row_number  rank  dense_rank
90           1     1           1
80           2     2           2
80           3     2           2
70           4     4           3


## 6.8 The Analyst's SQL Toolbox: Medians, Date Spines, Dedupe, Pivots and Checks

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers(cust_id INT, name TEXT, city TEXT);
INSERT INTO customers VALUES (1,'Asha','Pune'),(2,'Ravi','Delhi'),(3,'Meera','Pune'),
                             (4,'Imran','Mumbai'),(5,'Kavya','Delhi'),(6,'Dev','Mumbai');
CREATE TABLE orders(order_id INT, cust_id INT, order_date TEXT, amount REAL, status TEXT);
INSERT INTO orders VALUES
 (1,1,'2025-01-06',1200,'delivered'),(2,1,'2025-01-25',800,'delivered'),
 (3,2,'2025-01-22',1500,'delivered'),(4,3,'2025-02-04',500,'returned'),
 (5,3,'2025-02-20',900,'delivered'),(6,1,'2025-02-11',2000,'delivered'),
 (7,4,'2025-02-15',700,'delivered'),(8,2,'2025-03-02',1100,'delivered'),
 (9,5,'2025-03-03',NULL,'pending'),(10,4,'2025-03-12',1300,'delivered'),
 (11,1,'2025-03-15',600,'delivered'),(12,5,'2025-03-20',2500,'delivered'),
 (13,3,'2025-03-22',400,'cancelled'),(14,2,'2025-04-02',1700,'delivered');
CREATE TABLE raw(cust_id INT, email TEXT, updated TEXT);
INSERT INTO raw VALUES (1,'a@x.com','2025-01-01'),(1,'a@new.com','2025-02-01'),
  (2,'r@x.com','2025-01-05'),(2,'r@x.com','2025-01-05'),(3,'m@x.com','2025-01-09');
""")
def q(sql):
    print(pd.read_sql(sql, con).to_string(index=False)); print()

# 1. median of all non-NULL amounts, and per status
q("""WITH r AS (SELECT status, amount,
                  ROW_NUMBER() OVER (PARTITION BY status ORDER BY amount) AS rn,
                  COUNT(*) OVER (PARTITION BY status) AS n
           FROM orders WHERE amount IS NOT NULL)
     SELECT status, AVG(amount) AS median FROM r WHERE rn IN ((n + 1) / 2, (n + 2) / 2) GROUP BY status""")
# 2. date spine: 1 to 7 March, zero-filled
q("""WITH RECURSIVE spine(d) AS (
        SELECT '2025-03-01' UNION ALL SELECT date(d, '+1 day') FROM spine WHERE d < '2025-03-07')
     SELECT s.d, COUNT(o.order_id) AS orders, COALESCE(SUM(o.amount), 0) AS revenue
     FROM spine s LEFT JOIN orders o ON o.order_date = s.d GROUP BY s.d""")
# 3. dedupe: find duplicate keys, keep the latest row
q("SELECT cust_id, COUNT(*) AS n FROM raw GROUP BY cust_id HAVING COUNT(*) > 1")
q("""SELECT cust_id, email, updated FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cust_id ORDER BY updated DESC) AS rn FROM raw)
     WHERE rn = 1""")
# 4. pivot with CASE: one column per month
q("""SELECT c.city,
            SUM(CASE WHEN substr(o.order_date, 1, 7) = '2025-01' THEN o.amount ELSE 0 END) AS jan,
            SUM(CASE WHEN substr(o.order_date, 1, 7) = '2025-02' THEN o.amount ELSE 0 END) AS feb,
            SUM(CASE WHEN substr(o.order_date, 1, 7) = '2025-03' THEN o.amount ELSE 0 END) AS mar
     FROM orders o JOIN customers c USING (cust_id) WHERE o.status = 'delivered'
     GROUP BY c.city ORDER BY c.city""")

   status  median
cancelled   400.0
delivered  1200.0
 returned   500.0

         d  orders  revenue
2025-03-01       0      0.0
2025-03-02       1   1100.0
2025-03-03       1      0.0
2025-03-04       0      0.0
2025-03-05       0      0.0
2025-03-06       0      0.0
2025-03-07       0      0.0

 cust_id  n
       1  2
       2  2

 cust_id     email    updated
       1 a@new.com 2025-02-01
       2   r@x.com 2025-01-05
       3   m@x.com 2025-01-09

  city    jan    feb    mar
 Delhi 1500.0    0.0 3600.0
Mumbai    0.0  700.0 1300.0
  Pune 2000.0 2900.0  600.0


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE customers(cust_id INT, name TEXT, city TEXT, signup_date TEXT);
INSERT INTO customers VALUES
 (1,'Asha','Pune','2025-01-05'),(2,'Ravi','Delhi','2025-01-20'),
 (3,'Meera','Pune','2025-02-03'),(4,'Imran','Mumbai','2025-02-14'),
 (5,'Kavya','Delhi','2025-03-01'),(6,'Dev','Mumbai','2025-03-10');
CREATE TABLE orders(order_id INT, cust_id INT, order_date TEXT, amount REAL, status TEXT);
INSERT INTO orders VALUES
 (1,1,'2025-01-06',1200,'delivered'),(2,1,'2025-01-25',800,'delivered'),
 (3,2,'2025-01-22',1500,'delivered'),(4,3,'2025-02-04',500,'returned'),
 (5,3,'2025-02-20',900,'delivered'),(6,1,'2025-02-11',2000,'delivered'),
 (7,4,'2025-02-15',700,'delivered'),(8,2,'2025-03-02',1100,'delivered'),
 (9,5,'2025-03-03',NULL,'pending'),(10,4,'2025-03-12',1300,'delivered'),
 (11,1,'2025-03-15',600,'delivered'),(12,5,'2025-03-20',2500,'delivered'),
 (13,3,'2025-03-22',400,'cancelled'),(14,2,'2025-04-02',1700,'delivered');
""")
sql = """
SELECT 'null_amount' AS check_name, COUNT(*) AS bad_rows FROM orders WHERE amount IS NULL
UNION ALL SELECT 'dup_order_id', COUNT(*) FROM
   (SELECT order_id FROM orders GROUP BY order_id HAVING COUNT(*) > 1)
UNION ALL SELECT 'orphan_cust_id', COUNT(*) FROM orders o
   LEFT JOIN customers c USING (cust_id) WHERE c.cust_id IS NULL
UNION ALL SELECT 'order_before_signup', COUNT(*) FROM orders o
   JOIN customers c USING (cust_id) WHERE o.order_date < c.signup_date
UNION ALL SELECT 'negative_amount', COUNT(*) FROM orders WHERE amount < 0
UNION ALL SELECT 'bad_status', COUNT(*) FROM orders
   WHERE status NOT IN ('delivered','returned','cancelled','pending')"""
print(pd.read_sql(sql, con).to_string(index=False))

         check_name  bad_rows
        null_amount         1
       dup_order_id         0
     orphan_cust_id         0
order_before_signup         0
    negative_amount         0
         bad_status         0


## 6.9 Expert pandas: Copy-on-Write, Nullable Types and Profiling a Big File

In [ ]:
import pandas as pd
df = pd.DataFrame({"city": ["Pune", "Delhi", "Pune"], "amount": [100, 200, 300]})

# 1. changing a derived Series does not touch the original
col = df["amount"]
col.iloc[0] = 999
print("original:", df["amount"].tolist(), "| derived:", col.tolist())

# 2. changing a filtered copy does not touch the original
sub = df[df["city"] == "Pune"]
sub.loc[sub.index[0], "amount"] = -1
print("after sub edit:", df["amount"].tolist())

# 3. chained assignment (two brackets) silently does nothing, so use one .loc
fixed = df.copy()
fixed.loc[fixed["city"] == "Pune", "amount"] = 0
print("with .loc:", fixed["amount"].tolist())

original: [100, 200, 300] | derived: [999, 200, 300]
after sub edit: [100, 200, 300]
with .loc: [0, 200, 0]


In [ ]:
import pandas as pd
s = pd.Series([1, 2, None])
print("default :", s.dtype, s.tolist())
n = pd.Series([1, 2, None], dtype="Int64")
print("nullable:", n.dtype, n.tolist(), "sum =", n.sum(), "mean =", n.mean())
b = pd.Series([True, None, False], dtype="boolean")
print("boolean :", b.tolist(), "| b OR True ->", (b | True).tolist())
t = pd.Series(["a", None])
print("text    :", t.dtype, t.isna().tolist())

default : float64 [1.0, 2.0, nan]
nullable: Int64 [1, 2, <NA>] sum = 3 mean = 1.5
boolean : [True, <NA>, False] | b OR True -> [True, True, True]
text    : str [False, True]


In [ ]:
import numpy as np, pandas as pd, tempfile, os
rng = np.random.default_rng(7)
n = 1_000_000
big = pd.DataFrame({
    "order_id": np.arange(n),
    "city": rng.choice(["Pune", "Delhi", "Mumbai", "Chennai", "Kolkata"], n),
    "status": rng.choice(["delivered", "returned", "cancelled"], n, p=[0.9, 0.06, 0.04]),
    "qty": rng.integers(1, 6, n),
    "amount": rng.gamma(2.0, 400.0, n).round(2),
})
path = os.path.join(tempfile.mkdtemp(), "orders_big.csv")
big.to_csv(path, index=False)
print("1. file on disk (MB):", round(os.path.getsize(path) / 1e6, 1))

df = pd.read_csv(path)
print("2. default load (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))
print("   dtypes:", df.dtypes.astype(str).to_dict())

slim = pd.read_csv(path, usecols=["city", "status", "qty", "amount"],
                   dtype={"city": "category", "status": "category",
                          "qty": "int8", "amount": "float32"})
print("3. slim load (MB):", round(slim.memory_usage(deep=True).sum() / 1e6, 1))

full = df.groupby("city")["amount"].sum()
parts = [c.groupby("city")["amount"].sum()
         for c in pd.read_csv(path, usecols=["city", "amount"], chunksize=250_000)]
chunked = pd.concat(parts).groupby(level=0).sum()
print("4. chunked equals full:", bool(np.allclose(full, chunked)))
print((full / 1e6).round(2).to_dict())

1. file on disk (MB): 32.8
2. default load (MB): 136.7
   dtypes: {'order_id': 'int64', 'city': 'str', 'status': 'str', 'qty': 'int64', 'amount': 'float64'}
3. slim load (MB): 7.0
4. chunked equals full: True
{'Chennai': 160.55, 'Delhi': 159.82, 'Kolkata': 160.18, 'Mumbai': 159.99, 'Pune': 160.24}


In [ ]:
import pandas as pd
orders = pd.DataFrame({"order_id": [1, 2, 3, 3], "amount": [100.0, 250.0, -5.0, 80.0],
                       "status": ["delivered", "returned", "delivered", "oops"]})
checks = {
    "order_id unique": orders["order_id"].is_unique,
    "amount not negative": bool((orders["amount"] >= 0).all()),
    "status in allowed set": bool(orders["status"].isin(["delivered", "returned", "cancelled"]).all()),
    "no missing amount": bool(orders["amount"].notna().all()),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
failed = [k for k, v in checks.items() if not v]
print("failed:", len(failed), "of", len(checks))

FAIL order_id unique
FAIL amount not negative
FAIL status in allowed set
PASS no missing amount
failed: 3 of 4


## Chapter recap

## Practice set

Problem 1 `b = np.array([[1, 2], [3, 4], [5, 6]])`. What are the shape and `b.sum(axis=0)`?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch06.html